# 🧪 Auditor BI-RADS — RigoBERTa Clinical (2025) · versión estabilizada

Experimento con **`IIC/RigoBERTa-Clinical`**, modelo clínico español de 2025, construido sobre **XLM-RoBERTa-large** (~550M parámetros) mediante preentrenamiento de dominio sobre el corpus ClinText-SP.

**Ajustes específicos para este modelo (según su ficha oficial):**
- Usa `AutoModelForSequenceClassification` (cabeza correcta para la arquitectura).
- **LR bajo (1e-5) + warmup 10% + gradient clipping:** los modelos *large* colapsan con LR alto; estos ajustes estabilizan el fine-tuning. (El intento previo con LR 3e-5 colapsó: F1≈0).
- Pocas épocas (8) con early stopping, como recomienda la ficha (~2 épocas óptimas).

**Nota sobre comparación:** un modelo *large* requiere su propio LR óptimo; usar el mismo LR que un modelo *base* no es comparación justa sino handicap. El resto del pipeline (datos reales, split, balanceo, test, semilla 42) se mantiene idéntico.

⚠️ Es un modelo grande: lento y pesado en MPS (16 GB). El progreso se imprime por batch para seguimiento.

## 1 · Configuración y librerías

In [1]:
import torch, torch.nn as nn, numpy as np, pandas as pd, random, sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
sys.path.append('..')
from src.preprocessing import limpiar_texto, MAX_LENGTH

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO = 'IIC/RigoBERTa-Clinical'   # SOTA clínico español 2025 (DeBERTa-based)
print(f"PyTorch: {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Modelo base: {MODELO}")

PyTorch: 2.12.0 | Dispositivo: mps
Modelo base: IIC/RigoBERTa-Clinical


## 2 · Carga del dataset curado

In [2]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
print(f"Informes: {len(df)} | Columnas: {df.shape[1]}")

Informes: 4357 | Columnas: 19


## 3 · División train / validación / prueba (solo observaciones)

In [3]:
X = df['auditor_input'].values
y = df['BI-RADS'].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=SEED, stratify=y_tv)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3051 | Val: 652 | Test: 654


## 4 · Pesos de clase

In [4]:
clases = np.unique(y_train)
pesos = compute_class_weight(class_weight='balanced', classes=clases, y=y_train)
peso_vec = np.ones(7, dtype=np.float32)
for c,w in zip(clases,pesos): peso_vec[c]=w
peso_tensor = torch.tensor(peso_vec, dtype=torch.float32).to(DEVICE)
print("Pesos por clase (0..6):", np.round(peso_vec,2))

Pesos por clase (0..6): [  0.64   1.04   0.24   7.15  12.11  36.32 145.29]


## 5 · Augmentación de clases minoritarias (SOLO train)

In [5]:
def aumentar_texto(texto):
    variaciones = [
        ('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
        ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
        ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
        ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
        ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
        ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
        ('se evidencia','se observa'),
    ]
    t=texto
    for o,r in random.sample(variaciones, min(3,len(variaciones))):
        if o in t: t=t.replace(o,r,1)
    return t

mask_min=np.isin(y_train,[3,4,5,6]); X_min,y_min=X_train[mask_min],y_train[mask_min]
textos_aug,labels_aug=[],[]
for txt,lab in zip(X_min,y_min):
    for _ in range(3): textos_aug.append(aumentar_texto(txt)); labels_aug.append(lab)
X_train_aug=np.concatenate([X_train,np.array(textos_aug)])
y_train_aug=np.concatenate([y_train,np.array(labels_aug)])
idx=np.random.permutation(len(X_train_aug)); X_train_aug,y_train_aug=X_train_aug[idx],y_train_aug[idx]
print(f"Train aumentado: {len(X_train_aug)} (sin fuga de val/test)")

Train aumentado: 3387 (sin fuga de val/test)


## 6 · Tokenizador (RigoBERTa Clinical)

In [6]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception as e:
    print("Si falla por el tokenizador, instala sentencepiece y reintenta:")
    print("  !pip install sentencepiece")
    raise
print("Tokenizador cargado.")
print(f"Tokens especiales: CLS={tokenizer.cls_token} SEP={tokenizer.sep_token}")

Tokenizador cargado.
Tokens especiales: CLS=<s> SEP=</s>


## 7 · Dataset y DataLoader

In [7]:
class BIRADSDataset(Dataset):
    def __init__(self,t,l,tok,ml=MAX_LENGTH): self.t=list(t); self.l=list(l); self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

BATCH=16
train_loader=DataLoader(BIRADSDataset(X_train_aug,y_train_aug,tokenizer),batch_size=BATCH,shuffle=True)
val_loader  =DataLoader(BIRADSDataset(X_val,y_val,tokenizer),batch_size=BATCH)
test_loader =DataLoader(BIRADSDataset(X_test,y_test,tokenizer),batch_size=BATCH)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

Batches -> train: 212 | val: 41 | test: 41


## 8 · Arquitectura del modelo

RigoBERTa Clinical + cabeza de clasificación de 7 salidas. Se usa la representación del token inicial ([CLS]).

In [8]:
# CORRECCIÓN: usar AutoModelForSequenceClassification, que implementa el
# pooling y la cabeza correctos para CADA arquitectura (DeBERTa/RoBERTa/etc).
# El enfoque manual last_hidden_state[:,0] NO es válido para DeBERTa (RigoBERTa).
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODELO, num_labels=7
).to(DEVICE)
print(f"Modelo cargado en {DEVICE} (AutoModelForSequenceClassification, num_labels=7)")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: IIC/RigoBERTa-Clinical
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado en mps (AutoModelForSequenceClassification, num_labels=7)


## 9 · Configuración del entrenamiento

Mismos hiperparámetros que el experimento ganador (LR 2e-5 recomendado para RigoBERTa, 15 épocas, dropout 0.5, pesos de clase).

In [9]:
# Ajustes de ESTABILIDAD para XLM-RoBERTa-large (RigoBERTa):
# - LR bajo (1e-5): los modelos 'large' colapsan con LR alto (3e-5).
# - WARMUP del 10%: imprescindible para estabilizar el fine-tuning de large.
# - Gradient clipping (en el bucle): evita divergencia.
# Estos son los ajustes recomendados por la ficha del modelo, no los del modelo base.
LR=1e-5; EPOCHS=8; PATIENCE=3
criterio=nn.CrossEntropyLoss(weight=peso_tensor)
optimizer=torch.optim.AdamW(model.parameters(), lr=LR)
total_steps=len(train_loader)*EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
print(f"LR={LR} (bajo, para large) | Warmup 10% | Épocas={EPOCHS} | Gradient clipping activo")

LR=1e-05 (bajo, para large) | Warmup 10% | Épocas=8 | Gradient clipping activo


## 10 · Entrenamiento

In [10]:
def correr_epoca(loader, entrenar=True):
    model.train() if entrenar else model.eval()
    preds,reales,tot=[],[],0.0
    with torch.set_grad_enabled(entrenar):
        for j,b in enumerate(loader):
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lab=b['labels'].to(DEVICE)
            if entrenar: optimizer.zero_grad()
            logits = model(input_ids=ids, attention_mask=mask).logits   # <- cabeza correcta
            loss=criterio(logits,lab)
            if entrenar:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # estabilidad
                optimizer.step(); scheduler.step()
            tot+=loss.item(); preds.extend(logits.argmax(1).cpu().numpy()); reales.extend(lab.cpu().numpy())
            if entrenar and (j+1)%50==0:
                print(f"      batch {j+1}/{len(loader)}", flush=True)   # progreso por batch
    return tot/len(loader), f1_score(reales,preds,average='macro',zero_division=0)

import os; os.makedirs('../results',exist_ok=True); RUTA='../results/mejor_auditor_rigobert.pt'
print("🏋️  Entrenando auditor (RigoBERTa Clinical) — versión corregida")
print("Época | Train Loss | Val Loss | Train F1 | Val F1")
mejor,sin=0.0,0
for ep in range(1,EPOCHS+1):
    tl,tf1=correr_epoca(train_loader,True); vl,vf1=correr_epoca(val_loader,False)
    m=""
    if vf1>mejor: mejor,sin=vf1,0; torch.save(model.state_dict(),RUTA); m="  <- mejor"
    else: sin+=1
    print(f"  {ep:2d}  |  {tl:.4f}  |  {vl:.4f}  |  {tf1:.4f}  |  {vf1:.4f}{m}", flush=True)
    if sin>=PATIENCE: print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val F1 = {mejor:.4f}")

🏋️  Entrenando auditor (RigoBERTa Clinical) — versión corregida
Época | Train Loss | Val Loss | Train F1 | Val F1
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   1  |  1.7172  |  1.1221  |  0.1162  |  0.2500  <- mejor
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   2  |  0.9801  |  0.8709  |  0.4473  |  0.4241  <- mejor
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   3  |  0.7648  |  0.8585  |  0.5913  |  0.4649  <- mejor
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   4  |  0.4879  |  1.0309  |  0.7055  |  0.4549
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   5  |  0.3116  |  1.6250  |  0.7560  |  0.4417
      batch 50/212
      batch 100/212
      batch 150/212
      batch 200/212
   6  |  0.1896  |  1.3980  |  0.8603  |  0.4627
Early stopping en época 6
🏆 Mejor Val F1 = 0.4649


## 11 · Evaluación en test

In [11]:
model.load_state_dict(torch.load(RUTA,map_location=DEVICE)); model.eval()
preds,reales=[],[]
with torch.no_grad():
    for b in test_loader:
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
        logits=model(input_ids=ids, attention_mask=mask).logits
        preds.extend(logits.argmax(1).cpu().numpy()); reales.extend(b['labels'].numpy())
acc=(np.array(preds)==np.array(reales)).mean(); f1m=f1_score(reales,preds,average='macro',zero_division=0)
print("="*55); print("📊 RigoBERTa CLINICAL — TEST SET"); print("="*55)
print(f"  Accuracy: {acc:.4f}  |  F1-macro: {f1m:.4f}"); print("="*55)
print(classification_report(reales,preds,target_names=[f'BI-RADS {i}' for i in range(7)],zero_division=0))

📊 RigoBERTa CLINICAL — TEST SET
  Accuracy: 0.7875  |  F1-macro: 0.4702
              precision    recall  f1-score   support

   BI-RADS 0       0.67      0.62      0.65       145
   BI-RADS 1       0.90      0.98      0.94        89
   BI-RADS 2       0.99      0.82      0.89       396
   BI-RADS 3       0.15      0.85      0.25        13
   BI-RADS 4       0.17      0.38      0.23         8
   BI-RADS 5       0.25      0.50      0.33         2
   BI-RADS 6       0.00      0.00      0.00         1

    accuracy                           0.79       654
   macro avg       0.45      0.59      0.47       654
weighted avg       0.87      0.79      0.82       654



## 12 · Comparación de los tres modelos clínicos

In [12]:
print("F1-macro en test (sin fuga, mismo split/semilla 42):")
print(f"   BETO (español general):       0.4773")
print(f"   RoBERTa clínico (2022):       0.5871")
print(f"   RigoBERTa Clinical (2025):    {f1m:.4f}")
mejor=max(0.4773,0.5871,f1m)
cual = 'RigoBERTa Clinical' if f1m==mejor else ('RoBERTa clínico' if 0.5871==mejor else 'BETO')
print(f"\n   Mejor modelo: {cual}")

F1-macro en test (sin fuga, mismo split/semilla 42):
   BETO (español general):       0.4773
   RoBERTa clínico (2022):       0.5871
   RigoBERTa Clinical (2025):    0.4702

   Mejor modelo: RoBERTa clínico
